# Fig. S7B: Gene expression changes associated with Chaphamaparvovirus infection in chicken liver samples

Prepare representative-gene expression and membership tables for `FigS7B_plot.R`.
Requires `selected_term2gene_chaphama.tsv`, a separately curated term–gene table.


In [1]:
import os
import pandas as pd
from collections import defaultdict

In [2]:
# Define paths
basedir = "/path/to/project"

# Output directory
plot_path = f"{basedir}/output/revise_final/FigS7B/"

# GSEA results
out_path = f"{basedir}/output/revise_final/Fig6C/gsea/"
input_folder_path = out_path
# Curated term–gene table supplied separately
gene_table_path = f"{basedir}/output/260818_revise/rev_Fig5IJ_chaphama_4_panel/heatmap_chaphama/chatgpt/selected_term2gene_chaphama.tsv"
# DESeq2 results
input_rna_path = f"{basedir}/output/revise_final/Fig6C/deseq2/"

os.makedirs(plot_path, exist_ok=True)
print("saving files to:", plot_path)


saving files to: /Users/coco/iLab/itolab_backup/backup-latest/Lab/projects/2507blastx/output/revise_final/FigS7B/


# Select genes

In [3]:
# Pool leading-edge membership across comparisons
ttg = defaultdict(list)

for dirpath, dirnames, filenames in os.walk(input_folder_path):
    for fname in filenames:
        if fname != "gseapy.gene_set.prerank.report.csv":
            continue

        # Read the GSEA results
        path = os.path.join(dirpath, fname)
        res = pd.read_csv(path)
        project = path.split("/")[-2].split("_")[0]
        print("\nprocessing...", project)
        print("size:", res.shape)
        print("#NaN:", res.isna().sum().sum())
        print("#uniq Term:", len(set(res["Term"].to_list())))
        
        # Select terms
        selected_terms = [
            "Epithelial Mesenchymal Transition",
            "G2-M Checkpoint",
            "Interferon Alpha Response",
            "Allograft Rejection",
            "Inflammatory Response",
            "IL-6/JAK/STAT3 Signaling",
            "WIELAND_UP_BY_HBV_INFECTION",
            "Xenobiotic Metabolism",
            "Bile Acid Metabolism",
            "Fatty Acid Metabolism",
            "Adipogenesis"
        ]

        # Collect unique leading-edge genes for each selected term
        for idx in res.index:
            term = res.loc[idx, "Term"]
            if term not in selected_terms:
                continue
            print(term)
            genes = res.loc[idx, "Lead_genes"].split(";")
            if len(genes) != len(set(genes)):
                print("genes not unique")
            for gene in genes:
                if gene not in ttg[term]:
                    ttg[term].append(gene)


processing... PRJNA577590
size: (51, 10)
#NaN: 0
#uniq Term: 51
G2-M Checkpoint
WIELAND_UP_BY_HBV_INFECTION
Interferon Alpha Response
Adipogenesis
IL-6/JAK/STAT3 Signaling
Xenobiotic Metabolism
Bile Acid Metabolism
Fatty Acid Metabolism
Epithelial Mesenchymal Transition
Inflammatory Response
Allograft Rejection

processing... Fujino
size: (51, 10)
#NaN: 0
#uniq Term: 51
Xenobiotic Metabolism
Bile Acid Metabolism
Fatty Acid Metabolism
Adipogenesis
G2-M Checkpoint
WIELAND_UP_BY_HBV_INFECTION
IL-6/JAK/STAT3 Signaling
Epithelial Mesenchymal Transition
Interferon Alpha Response
Allograft Rejection
Inflammatory Response

processing... PRJNA612882
size: (51, 10)
#NaN: 0
#uniq Term: 51
Interferon Alpha Response
Epithelial Mesenchymal Transition
Xenobiotic Metabolism
Allograft Rejection
G2-M Checkpoint
Bile Acid Metabolism
Inflammatory Response
WIELAND_UP_BY_HBV_INFECTION
Fatty Acid Metabolism
IL-6/JAK/STAT3 Signaling
Adipogenesis

processing... PRJNA622813
size: (51, 10)
#NaN: 0
#uniq Term: 5

In [4]:
# Create the term–gene table
df = pd.DataFrame(
    [(k, v) for k, vs in ttg.items() for v in vs],
    columns=["term", "gene"]
)

df.to_csv(plot_path + "term2gene_chaphama.tsv" ,sep="\t")
# Preview results
print(df.shape)
print(df.isna().sum().sum())
df.head(3)

(1052, 2)
0


,term,gene
0,G2-M Checkpoint,RPA2
1,G2-M Checkpoint,ABL1
2,G2-M Checkpoint,ODF2


# Create heatmap tables

In [5]:
# Preview the selected gene table
genedf = pd.read_table(gene_table_path)
genedf

,term,gene
0,Epithelial Mesenchymal Transition,COL1A1
1,Epithelial Mesenchymal Transition,FN1
2,Epithelial Mesenchymal Transition,SNAI2
3,Epithelial Mesenchymal Transition,TGFB1
4,Epithelial Mesenchymal Transition,VIM
5,G2-M Checkpoint,AURKA
6,G2-M Checkpoint,CDC20
7,G2-M Checkpoint,CDK1
8,G2-M Checkpoint,PLK1
9,G2-M Checkpoint,TOP2A


In [6]:
# Check selected genes against the leading-edge genes
# Pool leading-edge membership across comparisons
test_dic = defaultdict(set)
all_genes = set()

# Read the selected gene table
genedf = pd.read_table(gene_table_path)
genes_oi = []
for g in genedf["gene"].to_list():
    if g not in genes_oi:
        genes_oi.append(g)

t_order = []
for t in genedf["term"].to_list():
    if t not in t_order:
        t_order.append(t)

for dirpath, dirnames, filenames in os.walk(input_folder_path):
    for fname in filenames:
        if fname != "gseapy.gene_set.prerank.report.csv":
            continue

        # Read the GSEA results
        path = os.path.join(dirpath, fname)
        res = pd.read_csv(path)
        project = path.split("/")[-2].split("_")[0]

        # Keep the selected terms
        for idx in res.index:
            term = res.loc[idx, "Term"]
            if term not in selected_terms:
                continue
            genes = res.loc[idx, "Lead_genes"].split(";")

            # Collect leading-edge genes
            for gene in genes:
                all_genes.add(gene)
                # Record term membership for selected genes
                if gene not in genes_oi:
                    continue
                test_dic[gene].add(term)

# Check that selected genes occur in the GSEA results
print(f"\n#all genes: {len(all_genes)}")
for goi in genes_oi:
    if goi not in sorted(list(all_genes)):
        print(f"### {goi} not exist! ###")

# Create gene–term associations
testdf = pd.DataFrame(
    [(k, v) for k, vs in test_dic.items() for v in sorted(list(vs))],
    columns=["gene", "term"]
)

# Create the gene–term membership matrix
term_bin = (pd.crosstab(testdf["gene"], testdf["term"]) > 0).astype(int)
term_bin = term_bin[t_order]
term_bin.columns = term_bin.columns.map(lambda x: x.replace("-", "_"))
# Order genes for the heatmap
term_bin = term_bin.sort_values(
            "gene",
            key=lambda s: pd.Categorical(s, categories=genes_oi, ordered=True)
        )
# Save tables
term_bin.to_csv(plot_path + "term_anno.csv")

# Preview results
term_bin.head(3)


#all genes: 888


term,Epithelial Mesenchymal Transition,G2_M Checkpoint,Interferon Alpha Response,Allograft Rejection,IL_6/JAK/STAT3 Signaling,Inflammatory Response,Bile Acid Metabolism,Xenobiotic Metabolism,Adipogenesis,Fatty Acid Metabolism,WIELAND_UP_BY_HBV_INFECTION
gene,,,,,,,,,,,
COL1A1,1,0,0,0,0,0,0,0,0,0,0
FN1,1,0,0,0,0,0,0,0,0,0,0
SNAI2,1,0,0,0,0,0,0,0,0,0,0


In [7]:
# Create the gene-by-project fold-change matrix
dfs = []
for dirpath, dirnames, filenames in os.walk(input_rna_path):
    for fname in filenames:
        if not(fname.endswith("_hs_gallus_and_Gene.txt")):
            continue

        # Read the DESeq2 results
        path = os.path.join(dirpath, fname)
        project = pd.read_table(path)
        proname = fname.split("_")[1]
        print("\nprocessing...", proname)

        # Keep the selected genes
        df_gene = project.loc[project["gene_name_hs"].isin(genes_oi), ["gene_name_hs", "log2FoldChange"]].reset_index(drop=True).copy()
        df_gene["Project"] = proname
        print(f"size for {proname}: {df_gene.shape}")

        # Collect results for this project
        dfs.append(df_gene)

# Combine results across projects
df_cat = pd.concat(dfs)
print("\nsize for all:", df_cat.shape)
print(df_cat.head(3))
# Pivot to a gene-by-project matrix
df_pivot = (df_cat.pivot_table(index="gene_name_hs", columns="Project", values="log2FoldChange").sort_index())
print("\nsize for df pivot:", df_pivot.shape)
print("#NaN for df pivot:", df_pivot.isna().sum().sum())
# Apply the selected gene order
df_pivor_sorted = df_pivot.sort_values(
            "gene_name_hs",
            key=lambda s: pd.Categorical(s, categories=genes_oi, ordered=True)
        )
print("size for sorted:", df_pivor_sorted.shape)
# Save tables
df_pivor_sorted.to_csv(plot_path + "gene_project_log2FC_matrix.csv")
# Preview results
df_pivor_sorted.head(3)


processing... Fujino
size for Fujino: (52, 3)

processing... PRJNA577590
size for PRJNA577590: (52, 3)

processing... PRJNA622813
size for PRJNA622813: (52, 3)

processing... PRJNA612882
size for PRJNA612882: (52, 3)

size for all: (208, 3)
  gene_name_hs  log2FoldChange Project
0         FGL2        0.115013  Fujino
1       AKR1D1       -4.325491  Fujino
2        USP18        1.834783  Fujino

size for df pivot: (52, 4)
#NaN for df pivot: 0
size for sorted: (52, 4)


Project,Fujino,PRJNA577590,PRJNA612882,PRJNA622813
gene_name_hs,,,,
COL1A1,4.541034,-0.021546,0.461615,-0.050896
FN1,-1.136427,0.878729,-0.066378,0.912364
SNAI2,0.018256,-1.302434,0.412287,0.587300
